<a href="https://colab.research.google.com/github/Qophy/PBML/blob/DataSet/data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import os
import sys
import tensorflow as tf
import pandas as pd
#import numpy as np
#import matplotlib.pyplot as plt
#import seaborn as sns


In [12]:
os.listdir('/content/drive/My Drive/code')

['INWSStaticDataGroupedByTimestamp1.xlsx',
 'pinns_tf2-main',
 'PINN-PDE(utube)',
 'PINNs-master(I)']

In [16]:
# Define the path to your modules directory on Google Drive
# Adjust 'My Drive/your_project_folder' if your modules are in a different location
data_path = '/content/drive/My Drive/code/'

# Add the modules_path to sys.path if it's not already there
if data_path not in sys.path:
    sys.path.append(data_path)

In [15]:
os.listdir(data_path)

['INWSStaticDataGroupedByTimestamp1.xlsx',
 'pinns_tf2-main',
 'PINN-PDE(utube)',
 'PINNs-master(I)']

In [20]:
dataset = pd.read_excel(data_path + 'INWSStaticDataGroupedByTimestamp1.xlsx')

In [ ]:
dataset.sample(10)

,Id,Node Id,Timestamp,Timestamp as DateTime,Temperature,Conductivity,pH,DissolvedOxygen
838,839,1,201506041004220000,2015/06/04 10:04:22 0000,17.82,330.5,8.10,45.9
19915,19916,1,201512240911559000,2015/12/24 09:11:55 9000,5.34,NaN,NaN,NaN
21425,21426,1,201512272322478000,2015/12/27 23:22:47 8000,NaN,NaN,NaN,6.4
23658,23659,2,201506012204410000,2015/06/01 22:04:41 0000,17.22,307.6,7.73,24.3
19473,19474,1,201512231524388000,2015/12/23 15:24:38 8000,NaN,NaN,NaN,147.4
28622,28623,2,201507211353280000,2015/07/21 13:53:28 0000,5.26,1276.9,70.16,0.3
18702,18703,1,201512220351014000,2015/12/22 03:51:01 4000,NaN,273.2,NaN,NaN
16814,16815,1,201511250755590000,2015/11/25 07:55:59 0000,7.62,256.0,7.87,102.7
6256,6257,1,201508141645120000,2015/08/14 16:45:12 0000,19.48,300.8,6.35,7.5
6261,6262,1,201508141736060000,2015/08/14 17:36:06 0000,19.53,303.6,6.48,8.0


## Find and Drop NaNs in the DATASET.

In [21]:
def print_rows_with_nans(df):
    nan_rows = df[df.isna().any(axis=1)]
    if nan_rows.empty:
        print(f"\n✅ No NaN values found in the dataset.")
    else:
        print(f"\n⚠️ Found {len(nan_rows)} rows with NaN values.")
        # print((nan_rows))

    return nan_rows

In [22]:
print_rows_with_nans(dataset)


⚠️ Found 4642 rows with NaN values.


,Id,Node Id,Timestamp,Timestamp as DateTime,Temperature,Conductivity,pH,DissolvedOxygen
0,1,1,201505011710380000,2015/05/01 17:10:38 0000,15.12,345.3,NaN,NaN
220,221,1,201505161519360000,2015/05/16 15:19:36 0000,16.06,370.1,NaN,69.5
376,377,1,201505291858129984,2015/05/29 18:58:13 0000,NaN,324.9,7.76,6.2
377,378,1,201505291902060000,2015/05/29 19:02:06 0000,14.74,325.0,NaN,5.7
3752,3753,1,201507140738500000,2015/07/14 07:38:50 0000,20.97,562.8,NaN,5.6
...,...,...,...,...,...,...,...,...
23069,23070,1,201601052108076992,2016-01-05 21:08:07,NaN,NaN,6.26,NaN
23070,23071,1,201601052108100000,2016/01/05 21:08:10 0000,NaN,NaN,NaN,9.4
23071,23072,1,201601052108100992,2016-01-05 21:08:10,NaN,257.2,NaN,NaN
23072,23073,1,201601052108124000,2016-01-05 21:08:12,2.59,NaN,NaN,NaN


In [23]:
nan_row_indices = print_rows_with_nans(dataset).index.tolist()


⚠️ Found 4642 rows with NaN values.


In [24]:
def drop_nan_rows(list_indices):
  dataset_shape = dataset.shape
  dataset_cleaned = dataset.drop(nan_row_indices)
  dataset_cleaned_shape = dataset_cleaned.shape

  return [dataset_cleaned, dataset_cleaned_shape, dataset_shape]

In [25]:
wq_data = drop_nan_rows(nan_row_indices)[0].drop(columns=["DissolvedOxygen", "Timestamp"])
print(wq_data.sample(5))

          Id  Node Id     Timestamp as DateTime  Temperature  Conductivity  \
15503  15504        1  2015/11/11 14:19:19 0000        10.35         274.1   
15968  15969        1  2015/11/15 18:14:26 0000         9.72         244.5   
11496  11497        1  2015/09/28 10:53:32 0000        13.70         299.4   
8958    8959        1  2015/09/07 05:53:06 0000        16.12         338.6   
10326  10327        1  2015/09/18 10:20:36 0000        16.69         332.0   

         pH  
15503  7.21  
15968  7.84  
11496  5.10  
8958   3.26  
10326  5.93  


## Cascading wq_data into two different measuring points

In [26]:
wq_data_node1 = wq_data[wq_data["Node Id"] == 1]
wq_data_node2 = wq_data[wq_data["Node Id"] == 2]

In [27]:
wq_data_node1.sample(5)


,Id,Node Id,Timestamp as DateTime,Temperature,Conductivity,pH
13504,13505,1,2015/10/23 02:29:45 0000,10.69,289.1,2.43
3412,3413,1,2015/07/10 04:12:18 0000,22.07,663.4,7.85
4353,4354,1,2015/07/18 13:36:08 0000,21.16,475.1,8.02
6339,6340,1,2015/08/15 07:09:38 0000,18.68,305.1,6.60
5104,5105,1,2015/07/24 01:46:09 0000,24.86,409.1,9.84


In [28]:
wq_data_node2.sample(5)

,Id,Node Id,Timestamp as DateTime,Temperature,Conductivity,pH
26986,26987,2,2015/06/30 09:00:26 0000,-0.35,970.8,-1.00
29699,29700,2,2015/08/29 09:05:06 0000,4.13,-5021.0,81.23
27207,27208,2,2015/07/01 22:04:32 0000,21.10,553.6,-13.78
28900,28901,2,2015/08/16 01:39:28 0000,-0.34,1822.0,-1.00
29144,29145,2,2015/08/17 22:16:42 0000,2.64,1076.7,92.67


# Create a FOLDER and name it PINN_WQS_dataset


In [29]:
#os.makedirs('/content/drive/My Drive/PINN_WQS_dataset', exist_ok=True)

In [30]:
#os.listdir('/content/drive/My Drive/PINN_WQS_dataset')

[]

# Save files to PINN_WQS_dataset directory

In [32]:
wq_data_node1.to_csv('/content/drive/My Drive/PINN_WQS_dataset/wq_dataset_node_1.csv', index=False)

In [33]:
wq_data_node2.to_csv('/content/drive/My Drive/PINN_WQS_dataset/wq_dataset_node_2.csv', index=False)